# 02 — Normalización de los datos experimentales

## Actividad 2.12

Esta actividad convierte los dos libros de Excel en dos tablas largas, una por protocolo. No las junta. No modifica `data/raw`. No imputa celdas vacías y no entrena ningún modelo.

Las unidades y las certezas salen de `docs/diccionario_variables.md`. La estructura de las hojas sale de `docs/auditoria_preparacion_datos.md`.

Al terminar quedan dos archivos:

- `data/processed/mapping_long.csv`
- `data/processed/surging_long.csv`

Un par numérico completo sigue siendo una fila en la que las dos mediciones del bloque se leen como números a la vez. Ese conteo no es una medición físicamente validada ni un experimento independiente.


## De dónde se leen los archivos

El cuaderno sube desde el directorio de trabajo hasta encontrar las dos copias. Así funciona si el kernel arranca en `notebooks/` o en la raíz del laboratorio. No se escribe la carpeta personal del computador.


In [1]:
import hashlib
import math
from pathlib import Path

import pandas as pd


def encontrar_raiz_laboratorio() -> Path:
    candidato = Path.cwd().resolve()
    for _ in range(6):
        mapping = candidato / "data" / "raw" / "mapping.xlsx"
        surging = candidato / "data" / "raw" / "surging.xlsx"
        if mapping.is_file() and surging.is_file():
            return candidato
        if candidato.parent == candidato:
            break
        candidato = candidato.parent
    raise FileNotFoundError(
        "No se encontró data/raw/mapping.xlsx junto a data/raw/surging.xlsx."
    )


def huella_sha256(ruta: Path) -> str:
    digest = hashlib.sha256()
    digest.update(ruta.read_bytes())
    return digest.hexdigest()


raiz = encontrar_raiz_laboratorio()
ruta_mapping = raiz / "data" / "raw" / "mapping.xlsx"
ruta_surging = raiz / "data" / "raw" / "surging.xlsx"
huellas_iniciales = {
    "mapping.xlsx": huella_sha256(ruta_mapping),
    "surging.xlsx": huella_sha256(ruta_surging),
}
huellas_publicadas = {
    "mapping.xlsx": "3b933ca84911484f5d912a44b025f5589abaf5ebc2b0426bf80c083869762bda",
    "surging.xlsx": "acdcc6c5fc3d8969280c8712c45657fbb863f163a60829584fece5abf75f28cd",
}
for nombre, huella in huellas_iniciales.items():
    if huella != huellas_publicadas[nombre]:
        raise AssertionError(f"La huella de {nombre} no coincide con la del laboratorio.")
    print(f"{nombre}: huella igual a la publicada")
print("Rutas relativas:", ruta_mapping.relative_to(raiz), "y", ruta_surging.relative_to(raiz))


mapping.xlsx: huella igual a la publicada
surging.xlsx: huella igual a la publicada
Rutas relativas: data/raw/mapping.xlsx y data/raw/surging.xlsx


## Cómo está guardada cada hoja

Los libros no son una tabla plana. La primera fila de cada franja es la condición experimental, y la siguiente nombra las variables. Por eso se leen con `header=None`.

Mapping tiene 18 columnas. Entre cada par `Flow rate` y `DP2-3` hay una columna vacía. Surging tiene 9 columnas y no usa ese hueco: la velocidad, `GVF0` y `DP2-3` van seguidos.

La actividad 2.2 ya midió el tamaño de las seis hojas. Aquí se vuelve a comprobar antes de mover cualquier número.


In [2]:
formas_esperadas = {
    "mapping": {"50psig": (50, 18), "100psig": (52, 18), "150psig": (55, 18)},
    "surging": {"50psig": (50, 9), "100psig": (61, 9), "150psig": (56, 9)},
}
libros = {
    "mapping": pd.ExcelFile(ruta_mapping),
    "surging": pd.ExcelFile(ruta_surging),
}
hojas = {}
for protocolo, libro in libros.items():
    if libro.sheet_names != ["50psig", "100psig", "150psig"]:
        raise AssertionError(f"{protocolo} no trae las tres hojas de presión.")
    for hoja in libro.sheet_names:
        tabla = pd.read_excel(libro, sheet_name=hoja, header=None)
        hojas[(protocolo, hoja)] = tabla
        if tabla.shape != formas_esperadas[protocolo][hoja]:
            raise AssertionError(
                f"{protocolo}/{hoja} mide {tabla.shape}, no {formas_esperadas[protocolo][hoja]}."
            )
        print(f"{protocolo}/{hoja}: {tabla.shape[0]} filas x {tabla.shape[1]} columnas")


mapping/50psig: 50 filas x 18 columnas
mapping/100psig: 52 filas x 18 columnas
mapping/150psig: 55 filas x 18 columnas
surging/50psig: 50 filas x 9 columnas
surging/100psig: 61 filas x 9 columnas
surging/150psig: 56 filas x 9 columnas


## Qué es una franja

Cada hoja tiene dos franjas. En mapping, la franja empieza en la fila cuyo texto es `Single Phase` o `Qgd=...`. La fila siguiente dice `Flow rate` y `DP2-3`. Los números empiezan una fila más abajo y terminan antes de la franja siguiente.

La velocidad no está repetida en cada fila. En mapping, el número ocupa la columna 0 de la primera fila de datos. En `50psig` esa celda vale 1800 o 3500. En `100psig` y `150psig` la misma celda está vacía.

En surging, la franja empieza en la fila de `0.75Qbep`, `Qbep` y `1.25Qbep`. El número de velocidad está en la fila de los rótulos `GVF0` y `DP2-3`, no en las filas de medición.


In [3]:
def es_numero(valor) -> bool:
    if isinstance(valor, bool) or valor is None:
        return False
    if isinstance(valor, (int, float)):
        return not (isinstance(valor, float) and math.isnan(valor))
    return False


def filas_de_franja_mapping(tabla: pd.DataFrame) -> list[int]:
    filas = []
    for fila in range(len(tabla)):
        for columna in range(tabla.shape[1]):
            valor = tabla.iat[fila, columna]
            if isinstance(valor, str) and (valor == "Single Phase" or valor.startswith("Qgd=")):
                filas.append(fila)
                break
    return filas


muestra = hojas[("mapping", "50psig")]
for fila in filas_de_franja_mapping(muestra):
    rotulos = [
        f"columna {columna}: {muestra.iat[fila, columna]}"
        for columna in range(muestra.shape[1])
        if isinstance(muestra.iat[fila, columna], str)
    ]
    print(f"fila {fila}:", "; ".join(rotulos))
print("Celda de velocidad de la primera franja, fila 2 columna 0:", muestra.iat[2, 0])
print("Celda de velocidad de la segunda franja, fila 24 columna 0:", muestra.iat[24, 0])
vacia = hojas[("mapping", "100psig")]
print("Mapping/100psig, misma posición de la primera franja:", vacia.iat[2, 0])
print("Mapping/100psig, misma posición de la segunda franja:", vacia.iat[22, 0])


fila 0: columna 1: Single Phase; columna 4: Qgd=0.005; columna 7: Qgd=0.01; columna 10: Qgd=0.015
fila 22: columna 1: Single Phase; columna 4: Qgd=0.01; columna 7: Qgd=0.02; columna 10: Qgd=0.03; columna 13: Qgd=0.04; columna 16: Qgd=0.05
Celda de velocidad de la primera franja, fila 2 columna 0: 1800
Celda de velocidad de la segunda franja, fila 24 columna 0: 3500
Mapping/100psig, misma posición de la primera franja: nan
Mapping/100psig, misma posición de la segunda franja: nan


## De dónde sale la velocidad cuando la celda está vacía

El diccionario no inventa la velocidad por el orden de las filas. La asigna el conjunto de rótulos Qgd, que es el mismo en las tres presiones:

- `Qgd=0.005`, `Qgd=0.01` y `Qgd=0.015`: 1800 rpm.
- `Qgd=0.01`, `Qgd=0.02`, `Qgd=0.03`, `Qgd=0.04` y `Qgd=0.05`: 3500 rpm.

Si la celda trae un número, tiene que ser ese mismo valor. Entonces `origen_velocidad` queda en `celda`. Si la celda está vacía, `velocidad_rpm` recibe el valor del diseño y `origen_velocidad` queda en `diseño_experimental`. La celda original no se rellena en el Excel.

`Single Phase` no trae un Qgd. La columna `qgd` de esa condición queda vacía. No se escribe un cero que el archivo no tiene.


In [4]:
CONJUNTO_1800 = frozenset({"Qgd=0.005", "Qgd=0.01", "Qgd=0.015"})
CONJUNTO_3500 = frozenset({"Qgd=0.01", "Qgd=0.02", "Qgd=0.03", "Qgd=0.04", "Qgd=0.05"})


def velocidad_segun_diseno(rotulos_qgd: set[str]) -> int:
    conjunto = frozenset(rotulos_qgd)
    if conjunto == CONJUNTO_1800:
        return 1800
    if conjunto == CONJUNTO_3500:
        return 3500
    raise ValueError("El conjunto de Qgd no está en el diseño experimental: " + ", ".join(sorted(conjunto)))


def psep_de_hoja(hoja: str) -> int:
    if not hoja.endswith("psig"):
        raise ValueError("La hoja no nombra la presión en psig.")
    return int(hoja.removesuffix("psig"))


def columnas_separadoras_vacias(tabla: pd.DataFrame) -> None:
    for columna in (3, 6, 9, 12, 15):
        if tabla[columna].notna().any():
            raise AssertionError(f"La columna separadora {columna} de mapping ya no está vacía.")


def normalizar_mapping(tabla: pd.DataFrame, hoja: str) -> tuple[pd.DataFrame, int]:
    columnas_separadoras_vacias(tabla)
    filas_franja = filas_de_franja_mapping(tabla)
    if len(filas_franja) != 2:
        raise ValueError(f"{hoja}: se esperaban dos franjas de mapping.")
    registros = []
    huecos = 0
    for indice, fila_condicion in enumerate(filas_franja):
        fila_encabezado = fila_condicion + 1
        fila_fin = filas_franja[indice + 1] if indice + 1 < len(filas_franja) else len(tabla)
        condiciones = []
        for columna in range(tabla.shape[1]):
            rotulo = tabla.iat[fila_condicion, columna]
            if isinstance(rotulo, str):
                condiciones.append((columna, rotulo))
        rotulos_qgd = [rotulo for _, rotulo in condiciones if rotulo.startswith("Qgd=")]
        velocidad_diseno = velocidad_segun_diseno(set(rotulos_qgd))
        fila_celda = fila_encabezado + 1
        celda = tabla.iat[fila_celda, 0]
        if es_numero(celda):
            if int(celda) != velocidad_diseno:
                raise ValueError(
                    f"{hoja}: la celda de velocidad no coincide con el diseño experimental."
                )
            origen = "celda"
            valor_celda = int(celda)
        elif pd.isna(celda):
            origen = "diseño_experimental"
            valor_celda = pd.NA
        else:
            raise ValueError(f"{hoja}: la celda de velocidad no es un número ni está vacía.")
        for columna, condicion in condiciones:
            if tabla.iat[fila_encabezado, columna] != "Flow rate":
                raise ValueError(f"{hoja}: falta el rótulo Flow rate.")
            if tabla.iat[fila_encabezado, columna + 1] != "DP2-3":
                raise ValueError(f"{hoja}: falta el rótulo DP2-3.")
            qgd = pd.NA if condicion == "Single Phase" else float(condicion.split("=", 1)[1])
            for fila in range(fila_celda, fila_fin):
                caudal = tabla.iat[fila, columna]
                incremento = tabla.iat[fila, columna + 1]
                hay_caudal = es_numero(caudal)
                hay_incremento = es_numero(incremento)
                if not hay_caudal and not hay_incremento:
                    huecos += 1
                    continue
                if hay_caudal != hay_incremento:
                    raise ValueError(
                        f"{hoja}, fila {fila}: el par está incompleto y no se imputa."
                    )
                registros.append(
                    {
                        "archivo": "mapping.xlsx",
                        "hoja": hoja,
                        "psep_psig": psep_de_hoja(hoja),
                        "psep_unidad": "psig",
                        "certeza_psep": "Alta",
                        "fila": fila,
                        "columna_condicion": columna,
                        "columna_caudal": columna,
                        "columna_dp": columna + 1,
                        "condicion": condicion,
                        "qgd": qgd,
                        "velocidad_rpm": velocidad_diseno,
                        "velocidad_unidad": "rpm",
                        "origen_velocidad": origen,
                        "fila_celda_velocidad": fila_celda,
                        "columna_celda_velocidad": 0,
                        "valor_celda_velocidad": valor_celda,
                        "certeza_velocidad": "Alta",
                        "ql_bpd": float(caudal),
                        "ql_unidad": "bpd",
                        "certeza_ql": "Alta, como interpretación",
                        "dp2_3_psi": float(incremento),
                        "dp2_3_unidad": "psi",
                        "certeza_dp": "Media",
                    }
                )
    return pd.DataFrame(registros), huecos


## Surging se lee con otra regla

El líquido no está en una columna de caudal. Está en el rótulo `0.75Qbep`, `Qbep` o `1.25Qbep`. No se multiplica esa fracción por 2700 bpd: ese caudal de mejor eficiencia está dado a 3500 rpm, y la tesis no escribe el equivalente a 1800 rpm.

`GVF0` se copia tal como está. No se divide por 100. La unidad de trabajo es el porcentaje, con certeza media, porque el sufijo 0 no está definido en la tesis.

La velocidad de surging sí está escrita. Las tres celdas de una misma franja tienen que coincidir. Si una faltara, no se copiaría desde la columna vecina.


In [5]:
FRACCION_QBEP = {"0.75Qbep": 0.75, "Qbep": 1.0, "1.25Qbep": 1.25}


def filas_de_franja_surging(tabla: pd.DataFrame) -> list[int]:
    filas = []
    for fila in range(len(tabla)):
        for columna in range(tabla.shape[1]):
            valor = tabla.iat[fila, columna]
            if isinstance(valor, str) and valor in FRACCION_QBEP:
                filas.append(fila)
                break
    return filas


def normalizar_surging(tabla: pd.DataFrame, hoja: str) -> tuple[pd.DataFrame, int]:
    filas_franja = filas_de_franja_surging(tabla)
    if len(filas_franja) != 2:
        raise ValueError(f"{hoja}: se esperaban dos franjas de surging.")
    registros = []
    huecos = 0
    for indice, fila_condicion in enumerate(filas_franja):
        fila_rotulos = fila_condicion + 1
        fila_fin = filas_franja[indice + 1] if indice + 1 < len(filas_franja) else len(tabla)
        condiciones = []
        for columna in range(tabla.shape[1]):
            rotulo = tabla.iat[fila_condicion, columna]
            if isinstance(rotulo, str):
                if rotulo == "Rotary Speed (rpm)":
                    continue
                if rotulo not in FRACCION_QBEP:
                    raise ValueError(f"{hoja}: condición de surging no reconocida.")
                condiciones.append((columna, rotulo))
        if [columna for columna, _ in condiciones] != [1, 4, 7]:
            raise ValueError(f"{hoja}: las condiciones de surging no están en las columnas 1, 4 y 7.")
        velocidades = []
        for columna, _ in condiciones:
            celda = tabla.iat[fila_rotulos, columna - 1]
            if not es_numero(celda):
                raise ValueError(f"{hoja}: falta la velocidad escrita de surging. No se imputa.")
            velocidades.append(int(celda))
        if len(set(velocidades)) != 1:
            raise ValueError(f"{hoja}: las tres celdas de velocidad de la franja no coinciden.")
        velocidad = velocidades[0]
        if velocidad not in (1800, 3500):
            raise ValueError(f"{hoja}: la velocidad escrita no es 1800 ni 3500 rpm.")
        for columna, condicion in condiciones:
            if tabla.iat[fila_rotulos, columna] != "GVF0":
                raise ValueError(f"{hoja}: falta el rótulo GVF0.")
            if tabla.iat[fila_rotulos, columna + 1] != "DP2-3":
                raise ValueError(f"{hoja}: falta el rótulo DP2-3.")
            for fila in range(fila_rotulos + 1, fila_fin):
                gvf = tabla.iat[fila, columna]
                incremento = tabla.iat[fila, columna + 1]
                hay_gvf = es_numero(gvf)
                hay_incremento = es_numero(incremento)
                if not hay_gvf and not hay_incremento:
                    huecos += 1
                    continue
                if hay_gvf != hay_incremento:
                    raise ValueError(
                        f"{hoja}, fila {fila}: el par de surging está incompleto y no se imputa."
                    )
                registros.append(
                    {
                        "archivo": "surging.xlsx",
                        "hoja": hoja,
                        "psep_psig": psep_de_hoja(hoja),
                        "psep_unidad": "psig",
                        "certeza_psep": "Alta",
                        "fila": fila,
                        "columna_condicion": columna,
                        "columna_gvf": columna,
                        "columna_dp": columna + 1,
                        "condicion": condicion,
                        "fraccion_qbep": FRACCION_QBEP[condicion],
                        "velocidad_rpm": velocidad,
                        "velocidad_unidad": "rpm",
                        "origen_velocidad": "celda",
                        "fila_celda_velocidad": fila_rotulos,
                        "columna_celda_velocidad": columna - 1,
                        "valor_celda_velocidad": velocidad,
                        "certeza_velocidad": "Alta",
                        "gvf0_porcentaje": float(gvf),
                        "gvf0_unidad": "porcentaje",
                        "certeza_gvf": "Media",
                        "dp2_3_psi": float(incremento),
                        "dp2_3_unidad": "psi",
                        "certeza_dp": "Media",
                    }
                )
    return pd.DataFrame(registros), huecos


## Construcción de las dos tablas

Cada fila de la tabla larga conserva el archivo, la hoja, la fila y las columnas del Excel, la condición y las dos mediciones. La fila se numera como en las actividades 2.1 a 2.11: la primera fila de la hoja es la fila 0.

Los huecos en los que caudal e incremento están vacíos a la vez no se convierten en filas. No son mediciones. Un par con una sola celda numérica detendría el cuaderno, porque rellenarlo sería imputar.


In [6]:
partes_mapping = []
huecos_mapping = {}
for hoja in ("50psig", "100psig", "150psig"):
    parte, huecos = normalizar_mapping(hojas[("mapping", hoja)], hoja)
    partes_mapping.append(parte)
    huecos_mapping[hoja] = huecos
mapping_largo = pd.concat(partes_mapping, ignore_index=True)

partes_surging = []
huecos_surging = {}
for hoja in ("50psig", "100psig", "150psig"):
    parte, huecos = normalizar_surging(hojas[("surging", hoja)], hoja)
    partes_surging.append(parte)
    huecos_surging[hoja] = huecos
surging_largo = pd.concat(partes_surging, ignore_index=True)

print("Filas de mapping:", len(mapping_largo))
print("Huecos vacíos omitidos en mapping:", huecos_mapping)
print("Filas de surging:", len(surging_largo))
print("Huecos vacíos omitidos en surging:", huecos_surging)
print("Origen de la velocidad en mapping:")
print(mapping_largo.groupby(["hoja", "origen_velocidad"], observed=True).size().to_string())


Filas de mapping: 570
Huecos vacíos omitidos en mapping: {'50psig': 50, '100psig': 60, '150psig': 58}
Filas de surging: 387
Huecos vacíos omitidos en surging: {'50psig': 19, '100psig': 42, '150psig': 17}
Origen de la velocidad en mapping:
hoja     origen_velocidad   
100psig  diseño_experimental    192
150psig  diseño_experimental    192
50psig   celda                  186


## Validación contra la actividad 2.10 y contra la auditoría

Mapping/`50psig` tiene que repetir 186 pares, 10 condiciones, 62 a 1800 rpm y 124 a 3500 rpm. Las otras cinco hojas tienen que repetir los totales exploratorios de la auditoría: 192, 192, 119, 129 y 139. Una diferencia se investiga; no se ajusta el conteo para que cuadre.


In [7]:
def pares_por_hoja(tabla: pd.DataFrame) -> dict[str, int]:
    return tabla.groupby("hoja", observed=True).size().to_dict()


auditoria_mapping = {"50psig": 186, "100psig": 192, "150psig": 192}
auditoria_surging = {"50psig": 119, "100psig": 129, "150psig": 139}
obtenido_mapping = pares_por_hoja(mapping_largo)
obtenido_surging = pares_por_hoja(surging_largo)
for hoja, esperado in auditoria_mapping.items():
    obtenido = int(obtenido_mapping[hoja])
    print(f"mapping/{hoja}: obtenido {obtenido}, auditoría {esperado}")
    if obtenido != esperado:
        raise AssertionError(f"mapping/{hoja} no reproduce la auditoría.")
for hoja, esperado in auditoria_surging.items():
    obtenido = int(obtenido_surging[hoja])
    print(f"surging/{hoja}: obtenido {obtenido}, auditoría {esperado}")
    if obtenido != esperado:
        raise AssertionError(f"surging/{hoja} no reproduce la auditoría.")

solo_50 = mapping_largo[mapping_largo["hoja"] == "50psig"]
condiciones_50 = solo_50.groupby(["velocidad_rpm", "condicion"], observed=True).ngroups
pares_1800 = int((solo_50["velocidad_rpm"] == 1800).sum())
pares_3500 = int((solo_50["velocidad_rpm"] == 3500).sum())
print(f"Mapping/50psig: {len(solo_50)} pares, {condiciones_50} condiciones, {pares_1800} a 1800, {pares_3500} a 3500")
if (len(solo_50), condiciones_50, pares_1800, pares_3500) != (186, 10, 62, 124):
    raise AssertionError("Mapping/50psig no reproduce 186 / 10 / 62 / 124.")

if set(mapping_largo.loc[mapping_largo["hoja"] == "50psig", "origen_velocidad"]) != {"celda"}:
    raise AssertionError("La velocidad de Mapping/50psig tenía que salir de la celda.")
diseno = mapping_largo[mapping_largo["hoja"].isin(["100psig", "150psig"])]
if set(diseno["origen_velocidad"]) != {"diseño_experimental"}:
    raise AssertionError("La velocidad de 100 y 150 psig tenía que quedar marcada como diseño.")
if diseno["valor_celda_velocidad"].notna().any():
    raise AssertionError("Una hoja sin velocidad escrita quedó con un número de celda.")
if set(surging_largo["origen_velocidad"]) != {"celda"}:
    raise AssertionError("La velocidad de surging tenía que salir de la celda.")
if mapping_largo["ql_bpd"].isna().any() or mapping_largo["dp2_3_psi"].isna().any():
    raise AssertionError("Entró una medición vacía a la tabla de mapping.")
if surging_largo["gvf0_porcentaje"].isna().any() or surging_largo["dp2_3_psi"].isna().any():
    raise AssertionError("Entró una medición vacía a la tabla de surging.")
if (surging_largo["gvf0_porcentaje"] > 1).sum() == 0:
    raise AssertionError("GVF0 no supera 1. La lectura en porcentaje quedaría sin apoyo en estos números.")
print("Validaciones de estructura, procedencia y conteos: coinciden.")
print(
    "GVF0 en porcentaje, mínimo y máximo:",
    float(surging_largo["gvf0_porcentaje"].min()),
    float(surging_largo["gvf0_porcentaje"].max()),
)
print(
    "QL en bpd, mínimo y máximo:",
    float(mapping_largo["ql_bpd"].min()),
    float(mapping_largo["ql_bpd"].max()),
)
print(
    "DP2-3 en psi, mapping, mínimo y máximo:",
    float(mapping_largo["dp2_3_psi"].min()),
    float(mapping_largo["dp2_3_psi"].max()),
)
print(
    "DP2-3 en psi, surging, mínimo y máximo:",
    float(surging_largo["dp2_3_psi"].min()),
    float(surging_largo["dp2_3_psi"].max()),
)


mapping/50psig: obtenido 186, auditoría 186
mapping/100psig: obtenido 192, auditoría 192
mapping/150psig: obtenido 192, auditoría 192
surging/50psig: obtenido 119, auditoría 119
surging/100psig: obtenido 129, auditoría 129
surging/150psig: obtenido 139, auditoría 139
Mapping/50psig: 186 pares, 10 condiciones, 62 a 1800, 124 a 3500
Validaciones de estructura, procedencia y conteos: coinciden.
GVF0 en porcentaje, mínimo y máximo: 0.0005573857518441711 19.141263668268486
QL en bpd, mínimo y máximo: 1.3529877894381905 4702.522256658226
DP2-3 en psi, mapping, mínimo y máximo: 0.08009041666666666 31.209657833333335
DP2-3 en psi, surging, mínimo y máximo: 0.3828433666666667 27.962517


## Escritura

Las tablas se guardan solo después de las comprobaciones. El Excel no se abre para escribir. Al final se vuelve a calcular la huella de los dos archivos de `data/raw`: tiene que ser la misma que al empezar.


In [8]:
destino = raiz / "data" / "processed"
destino.mkdir(parents=True, exist_ok=True)
ruta_mapping_largo = destino / "mapping_long.csv"
ruta_surging_largo = destino / "surging_long.csv"
enteros = [
    "psep_psig",
    "fila",
    "columna_condicion",
    "columna_dp",
    "velocidad_rpm",
    "fila_celda_velocidad",
    "columna_celda_velocidad",
    "valor_celda_velocidad",
]
for tabla in (mapping_largo, surging_largo):
    for columna in enteros:
        if columna in tabla.columns:
            tabla[columna] = tabla[columna].astype("Int64")
mapping_largo["columna_caudal"] = mapping_largo["columna_caudal"].astype("Int64")
surging_largo["columna_gvf"] = surging_largo["columna_gvf"].astype("Int64")

mapping_largo.to_csv(ruta_mapping_largo, index=False, na_rep="")
surging_largo.to_csv(ruta_surging_largo, index=False, na_rep="")

releido_mapping = pd.read_csv(ruta_mapping_largo)
releido_surging = pd.read_csv(ruta_surging_largo)
if len(releido_mapping) != len(mapping_largo) or len(releido_surging) != len(surging_largo):
    raise AssertionError("El CSV releído no conserva el número de filas.")
if list(releido_mapping["archivo"].unique()) != ["mapping.xlsx"]:
    raise AssertionError("La tabla de mapping contiene otro archivo.")
if list(releido_surging["archivo"].unique()) != ["surging.xlsx"]:
    raise AssertionError("La tabla de surging contiene otro archivo.")
if "gvf0_porcentaje" in releido_mapping.columns or "ql_bpd" in releido_surging.columns:
    raise AssertionError("Una tabla larga recibió columnas del otro protocolo.")

for nombre, ruta in (("mapping.xlsx", ruta_mapping), ("surging.xlsx", ruta_surging)):
    if huella_sha256(ruta) != huellas_iniciales[nombre]:
        raise AssertionError(f"{nombre} cambió durante la normalización.")
print("CSV escritos:")
print("-", ruta_mapping_largo.relative_to(raiz), f"({len(releido_mapping)} filas)")
print("-", ruta_surging_largo.relative_to(raiz), f"({len(releido_surging)} filas)")
print("Los Excel de data/raw conservan su huella.")
print("Verificaciones 186 / 10 / 62 / 124: coinciden.")


CSV escritos:
- data/processed/mapping_long.csv (570 filas)
- data/processed/surging_long.csv (387 filas)
Los Excel de data/raw conservan su huella.
Verificaciones 186 / 10 / 62 / 124: coinciden.


## Qué no hace esta actividad

No se calcula QG. Qmax está dado a 3500 rpm y no hay otro número para 1800 rpm. No se pasa `GVF0` a una fracción entre 0 y 1. No se convierten los psi a pascales ni los bpd a m³/s.

`DP2-3` sigue siendo una interpretación de certeza media: el rótulo no está en la tesis, y la unidad psi se toma del eje de esas curvas y del orden de los números. `Flow rate` se lee en bpd con certeza alta como interpretación, porque el encabezado no imprime la unidad. La velocidad de Mapping/`100psig` y Mapping/`150psig` no estaba en la celda.

Los puntos de una misma curva no son ensayos independientes. Estas tablas no eligen un reparto de entrenamiento y validación. El modelo, si se autoriza, queda para la actividad 2.13.
